In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


# **Ensemble Learning**

Ensemble Learning é o nome dado para a técnica de se utilizar varios preditores juntos em um grupo chamado ensemble em que trabalhem juntos de alguma forma/metodo (ensemble method) para melhorar a acurácia. Toda técnica de ensemble tem premissa de independencia entre individuos do grupo pois a ideia é que eles aprendam coisas diferentes para se complementarem, ou seja, um mitigar o erro do outro.

## **Voting Ensemble Method** 

Essa técnica de ensemble consiste em treinar um grupo de modelos independentes, ou seja, dados diferentes, algoritmos de treino diferentes ou hyperparametros diferentes. Na inferencia usamos o sistema de votos para entre os modelos para gerar o resultado, existem 2 formas de fazer isso:

- **Hard Voting**

Cada modelo faz uma predicao escolhendo uma classe e pegamos a classe que mais aparece entre o grupo de classificadores

- **Soft Voting**

Ao invés da classe, pegamos as *probabilidades das classes de todos*, somamos e tiramos a media de cada classe, pegamos a *classe com maior media de probabilidade* (Para usar soft todos os modelos tem que conseguir prever probabilidade). Isso é util pois permite que predicoes com *maior confianca recebam mais peso* pois a media será maior e além disso permite aplicacao em regressao em que podemos calcular a media das predicoes.


In [2]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42)

voting_clf = VotingClassifier(estimators=[
    ("lr", LogisticRegression()),
    ("rf", RandomForestClassifier()),
    ("svc", SVC())], voting="hard")

voting_clf.fit(X_train, y_train)

for clf in (LogisticRegression(), RandomForestClassifier(), SVC()):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, clf.score(X_test, y_test))

print("Voting Classifier", voting_clf.score(X_test, y_test))


    

LogisticRegression 0.864
RandomForestClassifier 0.896
SVC 0.896
Voting Classifier 0.896


## **Bagging e Pasting**

Uma forma outra forma de fazer um ensemble é treinar o mesmo algoritmo usando diferentes amostragens dos dados, isso pode ser feito de duas formas:

- **Bagging:** Cada modelo é treinado com uma amostra de mesmo tamanho do original gerada usando bootstrap nos dados originais, ou seja, usando amostragem com reposicao.
- **Pasting:** Mesma logica do bagging porém com amostragem sem reposicao.

Uma vez treinados a inferencia é feita de forma parecida com o hard voting ou soft voting. Uma outra vantagem de ensembles é que geralmente os preditores podem ser *treinados em paralelo e a inferencia também* pode ser feita em paralelo e isso faz eles escalarem muito bem.

Ensemble tende a *reduzir a variancia* dos modelos enquanto mantem o *vies bem estavel*, isso acontece pois a variancia, que é o erro aleatorio nas predicoes que o modelo aprendeu no treino devido a sua sensibilidade alta a variacoes, se cancela com o erro de outros modelos treinados com algoritmos/dados diferentes. O bias por outro lado, que é a tendencia/capacidade do algortimo para aprender padroes continua constante. 

- Por esse motivo existe ensemble de arvores que tem varianca alta pois tem capacidade/sensibilidade de overfittar facilmente e vies baixo pois conseguem aprender padroes complexos e nao lineares. 
- Por outro lado nao existe de regressoes lineares que tem alto vies devido a assumir linearidade e baixa variancia pois nao tem tanta capacidade de se adaptar ao ruido.

Bagging pode reduzir a variancia do modelo pois o *bootstrap pode ajudar a cancelar os erros* enquanto o pasting pode ser mais *barato computacionalmente e funciona bem se a variancia ja for baixa*.

### **Out-of-bag evaluation**

Durante o bagging, a amostragem do bootstrap para cada preditor costuma deixar cerca de 37% das instancias originais sem nunca ser amostrada para o preditor. Esse grupo de amostras que o preditor nunca viu sao chamadas de dados out-of-bag e podem ser usados como uma estimativa de como o ensemble vai se comportar com dados nunca vistos. Isso é chamado de out-of-bag evaluation.

### **Random Patches e Random Subspaces**

Bagging suporta também bootstrap de features ao invés de apenas dos dados e isso é particularmente util ao lidar com problemas de alta cardinalidade.

- **Random Patches:** Usar bootstrap tanto nas features como nas linhas
- **Random Subspaces:** Usar bootstrap apenas nas features

### **Random Forest**

Random forest é uma implementacao famosa de ensemble do tipo bagging em que nao apenas os dados de uma arvore de decisao sao aleatorios mas também as features que serao testadas nos splits (o padrao é das N features totais pegar raiz de N aleatorias).

- Existe a possibilidade de aleatorizar também os splits das features junto, tornando a *arvore extremamente aleatoria* pois existira aleatoriedade em 3 pontos diferentes: dados, features e splits
- O random forest tem seu proprio metodo de feature importance em que a importancia de uma feature *X é dada pela media da reducao na impureza de nós que usam X como feature ponderada pela quantidade de dados que passam ali*. Depois todas as importances sao somadas e é usado um valor entre [0-1] sendo a proporcao dessa soma a importancia.


In [3]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=1000,max_samples=100, n_jobs=-1, random_state=42)

bag_clf.fit(X_train, y_train)

print("Bagging Classifier", bag_clf.score(X_test, y_test))

random_clf = RandomForestClassifier(n_estimators=1000, max_leaf_nodes=16, n_jobs=-1, random_state=42)
random_clf.fit(X_train, y_train)

print("Random Forest Classifier", random_clf.score(X_test, y_test))

Bagging Classifier 0.904
Random Forest Classifier 0.92


# **Boosting**

Boosting é uma técnica de *ensemble* que treina vários estimadores **sequencialmente**, onde cada um tenta corrigir os erros do anterior. Por sua natureza sequencial, boosting **não é paralelizável** e escala pior que métodos como *bagging*/Random Forest.

Os dois principais métodos são **AdaBoost** e **Gradient Boosting**.

## **AdaBoost**

Treina um modelo, verifica onde ele errou, **aumenta o peso das instâncias erradas** e treina o próximo modelo com esses novos pesos. Depois que todos os modelos são treinados, a inferência é feita com uma **votação ponderada** pela confiança de cada estimador.

### **Algoritmo**

1. **Peso inicial uniforme**
   Todas as instâncias começam com peso $w^{(i)} = \frac{1}{m}$, onde `m` é o número de instâncias — ou seja, todas pesam igual.

2. **Treino e cálculo do erro**
   Treina-se o modelo e calcula-se seu **erro ponderado**: a soma dos pesos das instâncias erradas, dividida pela soma dos pesos de todas as instâncias. Isso indica quanto do peso total o modelo errou.

3. **Peso de confiança do modelo ($\alpha_j$)**
   Usando o log da razão $(1-r_j)/r_j$ (uma espécie de "inverso" do erro em escala logarítmica), multiplicado pela taxa de aprendizado ($\eta$), calcula-se a confiança daquele modelo na hora da inferência:
   - Erro alto → pouca influência.
   - Erro baixo → muita influência.
   - $\eta$ controla o quanto cada preditor pode pesar no total.

4. **Atualização dos pesos das instâncias**
   Instâncias acertadas mantêm o mesmo peso; instâncias erradas têm o peso **aumentado** (depois tudo é renormalizado para somar 1).

5. **Loop**
   Repete o processo até atingir o número de preditores desejado (ou até achar um preditor perfeito).

### **Inferência**

Votação ponderada pela confiança ($\alpha_j$) de cada modelo — o voto não vale 1, vale $\alpha_j$.

> Exemplo: se 2 modelos com confiança 3.0 votam na classe 1, o total da classe 1 é 6. A classe com maior soma de votos ponderados vence.

### **Observações**

- A generalização para múltiplas classes se chama **SAMME**, padrão no Scikit-Learn.
- Se o AdaBoost estiver **overfitando**, reduza o número de estimadores ou regularize mais os estimadores base.


## **Gradient Boosting**

Também sequencial, mas em vez de reponderar instâncias, cada modelo **prevê o erro (resíduo) do modelo anterior**. Somando todas as previsões, chegamos perto do valor correto.

> **Exemplo:** y real = 10.
> - Modelo 1 prevê 5 → erro de 5.
> - Modelo 2 é treinado para prever esse erro (5) e prevê 4 → erro de 1.
> - Modelo 3 é treinado para prever 1 e prevê 0.7 → erro de 0.3.
> - Soma final: 5 + 4 + 0.7 = **9.7**, bem próximo do valor real (10).

### **Algoritmo**

1. Treina o modelo nos dados de treino com os rótulos atuais.
2. Faz *predict* desse mesmo modelo e calcula o erro entre os rótulos corretos e os previstos (escalado pelo *learning rate*).
3. Usa esse erro como novo rótulo para o próximo modelo.
4. Repete até convergir ou atingir o número de estimadores.

### **Inferência**

Soma ponderada pelo *learning rate* das previsões de todos os modelos.

### **Early stopping**

O hiperparâmetro `n_iter_no_change` define o número máximo de árvores a adicionar sem melhoria — funciona como *early stopping*.

## **Histogram-Based Gradient Boosting (HGB)**

A ideia principal é acelerar o treino de árvores de decisão **discretizando** features contínuas em poucos valores inteiros (*bins*/*buckets*).

### **Por que fazer isso?**

Numa árvore comum, a complexidade é dominada pela **ordenação** dos valores contínuos para calcular os splits, repetida em cada nível de profundidade da árvore:

$$O(n \times m \log m)$$

(o $\log m$ é a profundidade de uma árvore crescida livremente, $\approx \log_2 m$)

Discretizando cada feature em `b` bins:

| Etapa | Complexidade |
|---|---|
| Construir o histograma | $O(n \times m)$ |
| Encontrar o melhor split, por nível, repetido em cada nível da árvore (profundidade `P`) | $O(n \times b \times P)$ |

**Total:** $O\big(n \times (m + b \times P)\big)$

Em boosting, `P` (`max_depth`) e `b` (`max_bins`) são hiperparâmetros fixos — não crescem com `m`. Por isso o termo $b \times P$ é dominado por `m`, e a complexidade se resume a:

$$O(n \times m)$$

Diferença central: a árvore comum ordena os valores brutos a cada split; o HGB reaproveita o histograma já pronto, então o custo por split não depende de `m`.

### **Trade-off**

Grande ganho de velocidade em datasets muito grandes, ao custo de qualidade nas predições — parte da informação é perdida no agrupamento em bins (efeito de regularização: pode ajudar contra overfitting ou causar underfitting, dependendo do dataset).

### **Bibliotecas** 

Existem várias implementações otimizadas de gradient boosting em Python: **LightGBM**, **CatBoost** e **XGBoost**.

In [21]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

ada_clf = AdaBoostClassifier(DecisionTreeClassifier(), n_estimators=100,learning_rate=0.7, random_state=42)
ada_clf.fit(X_train, y_train)

print("AdaBoost Classifier", ada_clf.score(X_test, y_test))

gb_clf = GradientBoostingClassifier(learning_rate=0.2, max_depth=3, random_state=42, n_iter_no_change=10)
gb_clf.fit(X_train, y_train)

print("Gradient Boosting Classifier", gb_clf.score(X_test, y_test))

hgb_clf = HistGradientBoostingClassifier(learning_rate=0.2, max_depth=3, random_state=42, early_stopping=True)
hgb_clf.fit(X_train, y_train)

print("HistGradientBoosting Classifier", hgb_clf.score(X_test, y_test))


AdaBoost Classifier 0.856
Gradient Boosting Classifier 0.904
HistGradientBoosting Classifier 0.88


## **Stacking** 

